# Compare SAY and Poisson likelihood (step2, flavor)
Nominal: SAY likelihood, `nbestfit50_all_param_2sigma_1D_10steps_PassingHESE` (as in `fit_result.ipynb`).
Alternative: identical fit settings with `llh: PoissonLLH` (`nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_Poisson`).

In [ ]:
import NNMFit
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
plotting_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind/studies/plots_compare_SAY_Poisson"
os.makedirs(plotting_path, exist_ok=True)

In [ ]:
import matplotlib.font_manager as font_manager
font_axis_label = {'family': 'serif',
        'color':  'black',
        'weight': 'normal',
        'size': 22,
        }
font_title = {'family': 'serif',
        'color':  'black',
        'weight': 'bold',
        'size': 20,
        }
font_legend = font_manager.FontProperties(family='serif',
                                   weight='normal',
                                   style='normal', size=13)

plt.rcParams["figure.figsize"] = (6,6)
plt.rcParams.update({'font.family':'serif'})

In [ ]:
def get_1sigma(scan, parameter, bestfit_val, threshold=1.0):
    """Find where dllh crosses threshold via sign-change detection + linear interpolation."""
    scan_sorted = scan.sort_values(parameter).reset_index(drop=True)
    x = scan_sorted[parameter].values
    delta = scan_sorted["dllh"].values - threshold

    sign_changes = np.where(np.diff(np.sign(delta)))[0]
    roots = []
    for idx in sign_changes:
        x0, x1 = x[idx], x[idx + 1]
        y0, y1 = delta[idx], delta[idx + 1]
        roots.append(x0 - y0 * (x1 - x0) / (y1 - y0))

    roots = np.array(roots)
    lower = roots[roots < bestfit_val]
    upper = roots[roots > bestfit_val]

    err_lo = bestfit_val - lower[-1] if len(lower) else np.nan
    err_hi = upper[0] - bestfit_val if len(upper) else np.nan
    return err_lo, err_hi

In [ ]:
data_fits = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor_reunblind/data_fits"
say_scan_dir = f"{data_fits}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE"
poisson_scan_dir = f"{data_fits}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_Poisson"

drop_unsuccessful = True  # as in fit_result.ipynb
scan_dict = {
    "SAY": NNMFit.utilities.ScanHandler(say_scan_dir, dump=False, drop_unsuccessful=drop_unsuccessful),
    "Poisson": NNMFit.utilities.ScanHandler(poisson_scan_dir, dump=False, drop_unsuccessful=drop_unsuccessful),
}
bestfits = {name: scan.get_freefit(drop_unsuccessful="True") for name, scan in scan_dict.items()}
colors = {"SAY": "steelblue", "Poisson": "darkorange"}

# absolute llh values of different likelihood definitions are not comparable,
# every profile is relative to the minimum of its own likelihood
for name, bf in bestfits.items():
    print(f"{name}: freefit llh = {bf['llh']:.4f}")

## Best fit and $1\sigma$ errors

In [ ]:
scan_parameters = [
    "astro_norm", "gamma_astro", "astro_nue_ratio", "astro_nutau_ratio",
    "conv_norm", "prompt_norm", "combined_muongun_norm", "muongun_asymmetry",
    "delta_gamma", "CR_grad", "barr_h", "barr_w", "barr_y", "barr_z",
    "dom_eff", "ice_scat", "ice_abs", "ice_crystal", "ice_holep0", "ice_holep1",
]

scans = {}   # scans[name][parameter] = profile dataframe
errors = {}  # errors[name][parameter] = (bestfit, err_lo, err_hi)
for name in scan_dict:
    scans[name], errors[name] = {}, {}
    for parameter in scan_parameters:
        try:
            scan = scan_dict[name].get_scan_df(param_name=parameter, drop_unsuccessful=drop_unsuccessful)
        except Exception as e:
            print(f"{name}: skipping {parameter}: {e}")
            continue
        bestfit_val = bestfits[name][parameter]
        scans[name][parameter] = scan
        errors[name][parameter] = (bestfit_val, *get_1sigma(scan, parameter, bestfit_val))

rows = {}
for parameter in scan_parameters:
    say_bf, say_lo, say_hi = errors["SAY"].get(parameter, (np.nan,) * 3)
    poi_bf, poi_lo, poi_hi = errors["Poisson"].get(parameter, (np.nan,) * 3)
    diff = poi_bf - say_bf
    say_err = say_hi if diff > 0 else say_lo
    rows[parameter] = {
        "SAY": say_bf, "SAY -1sig": say_lo, "SAY +1sig": say_hi,
        "Poisson": poi_bf, "Poisson -1sig": poi_lo, "Poisson +1sig": poi_hi,
        "Poisson - SAY": diff, "diff / SAY 1sig": diff / say_err,
    }
df_compare = pd.DataFrame(rows).T
with pd.option_context("display.float_format", "{:.3f}".format):
    display(df_compare)

In [ ]:
# best fit flavor composition at Earth, nu_e : nu_mu : nu_tau
for name, bf in bestfits.items():
    norm = 1 + bf["astro_nue_ratio"] + bf["astro_nutau_ratio"]
    print(f"{name}: {bf['astro_nue_ratio'] / norm:.3f} : {1 / norm:.3f} : {bf['astro_nutau_ratio'] / norm:.3f}")

## Signal profiles

In [ ]:
parameters = ["astro_norm", "gamma_astro", "astro_nue_ratio", "astro_nutau_ratio"]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for ax, parameter in zip(axes, parameters):
    for name in scan_dict:
        scan_sorted = scans[name][parameter].sort_values(parameter)
        bestfit_val, err_lo, err_hi = errors[name][parameter]
        print(f"{name} | {parameter}: {bestfit_val:.4g} +{err_hi:.4g} / -{err_lo:.4g}")

        ax.plot(scan_sorted[parameter], scan_sorted["dllh"], color=colors[name], linewidth=2,
                label=f"{name}: ${bestfit_val:.3f}^{{+{err_hi:.3f}}}_{{-{err_lo:.3f}}}$")
        ax.axvline(bestfit_val, color=colors[name], linewidth=1.5, linestyle="--")
        ax.axvline(bestfit_val - err_lo, color=colors[name], linestyle=":", linewidth=1.2)
        ax.axvline(bestfit_val + err_hi, color=colors[name], linestyle=":", linewidth=1.2)

    ax.axhline(1.0, color="black", linestyle="--", linewidth=1.2)
    ax.set_xlabel(parameter, fontdict=font_axis_label)
    ax.set_ylabel(r"$-2\Delta\mathrm{log}\mathcal{L}$", fontdict=font_axis_label)
    ax.set_ylim(-0.5, 5)
    ax.legend(prop=font_legend)

plt.tight_layout()
plt.savefig(f"{plotting_path}/compare_signal_scan_SAY_Poisson.png", bbox_inches="tight")
plt.show()

## All parameter profiles

In [ ]:
ncols = 4
nrows = int(np.ceil(len(scan_parameters) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
axes = axes.flatten()

for ax, parameter in zip(axes, scan_parameters):
    for name in scan_dict:
        if parameter not in scans[name]:
            continue
        scan_sorted = scans[name][parameter].sort_values(parameter)
        bestfit_val, err_lo, err_hi = errors[name][parameter]
        ax.plot(scan_sorted[parameter], scan_sorted["dllh"], color=colors[name], linewidth=2,
                label=f"{name}: {bestfit_val:.3g}")
        ax.axvline(bestfit_val, color=colors[name], linewidth=1.5, linestyle="--")
        if np.isfinite(err_lo):
            ax.axvline(bestfit_val - err_lo, color=colors[name], linestyle=":", linewidth=1.2)
        if np.isfinite(err_hi):
            ax.axvline(bestfit_val + err_hi, color=colors[name], linestyle=":", linewidth=1.2)

    ax.axhline(1.0, color="black", linestyle="--", linewidth=1.2)
    ax.set_xlabel(parameter, fontdict=font_axis_label)
    ax.set_ylabel(r"$-2\Delta\mathrm{log}\mathcal{L}$", fontdict=font_axis_label)
    ax.set_ylim(-0.5, 5)
    ax.legend(prop=font_legend)

for ax in axes[len(scan_parameters):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(f"{plotting_path}/all_param_scan_SAY_Poisson.png", bbox_inches="tight")
plt.show()

## Overview: best fit $\pm 1\sigma$ relative to the SAY best fit

In [ ]:
fig, ax = plt.subplots(figsize=(7, 0.55 * len(scan_parameters) + 2))
ypos = np.arange(len(scan_parameters))[::-1]

for offset, name in zip([0.15, -0.15], scan_dict):
    bf = np.array([errors[name].get(p, (np.nan,) * 3)[0] for p in scan_parameters]) - df_compare["SAY"].values.astype(float)
    lo = np.array([errors[name].get(p, (np.nan,) * 3)[1] for p in scan_parameters])
    hi = np.array([errors[name].get(p, (np.nan,) * 3)[2] for p in scan_parameters])
    # missing (open) errors are drawn as zero length
    ax.errorbar(bf, ypos + offset, xerr=[np.nan_to_num(lo), np.nan_to_num(hi)],
                fmt="o", color=colors[name], elinewidth=2, capsize=4, markersize=5, label=name)

ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_yticks(ypos)
ax.set_yticklabels(scan_parameters)
ax.set_xlabel("deviation from SAY best fit", fontdict=font_axis_label)
ax.grid(axis="x", alpha=0.3)
ax.legend(prop=font_legend)

plt.tight_layout()
plt.savefig(f"{plotting_path}/errors_overview_SAY_Poisson.png", bbox_inches="tight")
plt.show()

## LaTeX table: SAY vs Poisson
Best fit and $1\sigma$ errors from the 1D profile scans above (`---`: profile does not cross 1 within the scan / physical bound).

In [ ]:
import sys
sys.path.insert(0, "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind")
from analysis_utils import latex_comparison_table

print(latex_comparison_table({"SAY": errors["SAY"], "Poisson": errors["Poisson"]},
                             caption=r"Best fit values with $1\sigma$ asymmetric uncertainties for the SAY and the Poisson likelihood.",
                             label="tab:bestfit_SAY_Poisson"))

## Flavor triangle: SAY vs Poisson
SAY: the three flavor scans `data_fits_flavor_triangle/nbestfit50_fine_{0,1,2}` combined (as in `flavor_triangle.ipynb`).
Poisson: `data_fits_flavor_triangle/nbestfit50_Poisson`, identical settings apart from `llh: PoissonLLH`.

In [ ]:
from analysis_utils import load_triangle_scan, combine_scans, flavor_fractions
sys.path.append("/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks")
from plot_utils_triangle import *

triangle_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor_reunblind/data_fits_flavor_triangle"
tri_scan_dirs = {
    "nbestfit50_fine_0": f"{triangle_path}/nbestfit50_fine_0",
    "nbestfit50_fine_1": f"{triangle_path}/nbestfit50_fine_1",
    "nbestfit50_fine_2": f"{triangle_path}/nbestfit50_fine_2",
    "Poisson":           f"{triangle_path}/nbestfit50_Poisson",
}
N_grid = 40
tri_scans, tri_data, tri_bestfits = {}, {}, {}
for name, scan_dir in tri_scan_dirs.items():
    tri_scans[name], tri_data[name], tri_bestfits[name] = load_triangle_scan(scan_dir, N_grid=N_grid)
    print(f"{name}: {len(tri_data[name]['ts'])} successful scan points, freefit llh = {tri_bestfits[name]['llh']:.4f}")

say_names = ["nbestfit50_fine_0", "nbestfit50_fine_1", "nbestfit50_fine_2"]
tri_data["SAY"], tri_bestfits["SAY"] = combine_scans(say_names, tri_data, tri_bestfits, N_grid)

for name in ["SAY", "Poisson"]:
    bf, fr = tri_bestfits[name], flavor_fractions(tri_bestfits[name])
    print(f"{name}: astro_nue_ratio = {bf['astro_nue_ratio']:.3f}, astro_nutau_ratio = {bf['astro_nutau_ratio']:.3f}, "
          f"nu_e : nu_mu : nu_tau = {fr['f_e']:.3f} : {fr['f_mu']:.3f} : {fr['f_tau']:.3f}")

In [ ]:
fr = flavor_fractions(tri_bestfits["Poisson"])
C = plot_contour(data=tri_data, name="Poisson", label="Poisson", levels=["68%", "90%", "95%"], title="",
                 scenarios=True, show_ts_labels=False, bestfit=[fr["f_tau"], fr["f_e"]],
                 savepath=f"{plotting_path}/flavor_triangle_Poisson.png")
plt.show()

In [ ]:
compare_names = ["SAY", "Poisson"]
compare_labels = ["SAY (3 scans combined)", "Poisson"]
C = compare_contours(tri_data, names=compare_names, labels=compare_labels,
                     levels=["68%", "90%", "95%"], title="", scenarios=True)
ax = plt.gca()
for i, name in enumerate(compare_names):
    fr = flavor_fractions(tri_bestfits[name])
    ax.scatter([fr["f_tau"]], [fr["f_e"]], marker="X", facecolor=colours[i], edgecolor="k", lw=0.5, s=80, zorder=10)
plt.savefig(f"{plotting_path}/flavor_triangle_SAY_vs_Poisson.png", bbox_inches="tight")
plt.show()